# Repairing Test Notebook
This notebook verifies the Repairer implementations (ILP and Vertex Cover variants), ensuring they correctly remove tuples to satisfy constraints while maintaining marginal frequencies.

## 1. Setup & Orchestration

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.loading.file_loader import FileLoader
from src.loading.components.data_loader import DataLoader
from src.loading.components.dcs_loader import DCsLoader
from src.loading.components.metadata_loader import MetadataLoader
from src.loading.components.data_encoder import DataEncoder
from src.loading.components.dcs_encoder import DCsEncoder
from src.synthesizing.co_noise import CoNoise
from src.marginals_obtaining.top_k_obtainer import TopKObtainer
from src.marginals_obtaining.utility_functions.distance_utility import DistanceUtility
from src.repairing.ilp_repairer import ILPRepairer
from src.repairing.weighted_vc_repairer import WeightedVCRepairer
from src.repairing.vanilla_vc_repairer import VanillaVCRepairer
from src.repairing.classic_vc_repairer import ClassicVCRepairer

def get_dataset(dataset_name):
    loader = FileLoader(
        name=dataset_name, 
        base_path="../data",
        data_loader=DataLoader(),
        dcs_loader=DCsLoader(),
        metadata_loader=MetadataLoader(),
        data_encoder=DataEncoder(),
        dcs_encoder=DCsEncoder()
    )
    return loader.load()

## 2. Prepare Data (Private -> Synthetic -> Marginals)
We use the `tax` dataset for a quick test.

In [ ]:
private_dataset = get_dataset("tax")
print(f"Private data violations: {len(private_dataset.get_violations())}")

synthesizer = CoNoise(num_of_iterations=50)
synthetic_dataset = synthesizer.synthesize(private_dataset)
print(f"Synthetic data violations: {len(synthetic_dataset.get_violations())}")

obtainer = TopKObtainer(selection_budget=1.0, generation_budget=1.0, k=10, utility_function=DistanceUtility())
marginals = obtainer.obtain(private_dataset, synthetic_dataset)
print(f"Obtained {len(marginals)} marginals.")

## 3. ILP Repair
Run the ILP repairer and check if violations are eliminated.

In [ ]:
print("Running ILP Repair...")
repairer_ilp = ILPRepairer(alpha=0.8, use_marginals=True, gurobi_params={"OutputFlag": 0, "TimeLimit": 60})
repaired_ilp = repairer_ilp.repair(synthetic_dataset, marginals)
print(f"  - Repaired size: {len(repaired_ilp.data)}")
print(f"  - Repaired violations: {len(repaired_ilp.get_violations())}")
assert len(repaired_ilp.get_violations()) == 0

## 4. Vertex Cover Repairs
Test Weighted, Vanilla, and Classic Vertex Cover implementations.

In [ ]:
repairers = [
    ("Weighted VC", WeightedVCRepairer(alpha=0.5)),
    ("Vanilla VC", VanillaVCRepairer()),
    ("Classic VC", ClassicVCRepairer())
]

results = {"ILP": repaired_ilp}

for name, repairer in repairers:
    print(f"Running {name}...")
    repaired = repairer.repair(synthetic_dataset, marginals)
    print(f"  - Repaired size: {len(repaired.data)}")
    print(f"  - Repaired violations: {len(repaired.get_violations())}")
    assert len(repaired.get_violations()) == 0, f"{name} failed to remove all violations!"
    results[name] = repaired

print("All Vertex Cover tests passed!")

## 5. Evaluation and Comparison

In [ ]:
def calculate_total_error(data, marginal_set):
    return sum(m.calculate_error(data) for m in marginal_set) / len(marginal_set)

print(f"Synthetic Data Error: {calculate_total_error(synthetic_dataset.data, marginals):.4f}")
for name, dataset in results.items():
    error = calculate_total_error(dataset.data, marginals)
    print(f"{name} Repaired Error: {error:.4f} | Size: {len(dataset.data)}")